# Frozen Digit-Addition Subunit Project


## Imports and Dataset

In [48]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time

torch.manual_seed(0)
print(f"PyTorch {torch.__version__}")

# One-hot encoding

def one_hot(digit: int, num_classes = 10) -> torch.Tensor:   #Return a 1-D float tensor of length num_classes (10) with a 1 at the digit.
    v = torch.zeros(num_classes)
    v[digit] = 1.0
    return v

def encode_pair(a: int, b: int) -> torch.Tensor:          #Concatenate one-hot encodings of a and b → 20-d vector.
    return torch.cat([one_hot(a), one_hot(b)])

# Dataset builders 

def build_100(): #All 100 (a, b) pairs. Labels: ones digit only.
    
    X, yo = [], []
    for a in range(10):
        for b in range(10):
            X.append(encode_pair(a, b))
            yo.append((a + b) % 10)
    return torch.stack(X), torch.tensor(yo, dtype=torch.long)

def build_200(): #All 200 (a, b, carry_in) triples. Labels: ones digit + carry out.
    X, yo, yc = [], [], []
    for a in range(10): #if a and b are digits 0-9
        for b in range(10): #if a and b are digits 0-9
            for cin in range(2): #if carry in is 0 or 1
                v = torch.zeros(22)
                v[a] = 1.0          # one-hot digit a
                v[10 + b] = 1.0     # one-hot digit b
                v[20 + cin] = 1.0   # one-hot carry in (2 classes)
                X.append(v)
                yo.append((a + b + cin) % 10)
                yc.append((a + b + cin) // 10)
    return torch.stack(X), torch.tensor(yo, dtype=torch.long), torch.tensor(yc, dtype=torch.long)

X100, Y_ones = build_100()
X200, Y_ones200, Y_carry200 = build_200()
print(f"Dataset shapes: X100={X100.shape} {Y_ones.shape} X200={X200.shape} {Y_ones200.shape} {Y_carry200.shape}")

PyTorch 2.12.0+cpu
Dataset shapes: X100=torch.Size([100, 20]) torch.Size([100]) X200=torch.Size([200, 22]) torch.Size([200]) torch.Size([200])


## Train & Freeze Model A (Ones-Digit MLP)

Given two digits a and b, predict (a + b) mod 10.

Architecture: Input(20) → Linear(64) → ReLU → Linear(64) → ReLU → Linear(10) → logits


In [49]:
class OnesDigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.h1  = nn.Linear(20, 64)
        self.h2  = nn.Linear(64, 64)
        self.out = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(F.relu(self.h2(F.relu(self.h1(x)))))


def train_model_a(seed: int = 0, max_epochs: int = 10_000, lr: float = 1e-2) -> OnesDigitMLP:
    #Train Model A to 100% accuracy, freeze, and return.
    torch.manual_seed(seed)
    model = OnesDigitMLP()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print("Training Model A...")
    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        loss = F.cross_entropy(model(X100), Y_ones)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            if model(X100).argmax(1).eq(Y_ones).all():
                print(f"  100% accuracy at epoch {epoch}")
                break

    # Freeze all parameters
    for p in model.parameters():
        p.requires_grad_(False)
    model.eval()
    return model

model_a = train_model_a()

# Verify
with torch.no_grad():
    preds = model_a(X100).argmax(1)
correct = preds.eq(Y_ones).sum().item()
trainable = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f"Accuracy     : {correct}/100")
print(f"Trainable params after freeze: {trainable}  (must be 0)")

Training Model A...
  100% accuracy at epoch 26
Accuracy     : 100/100
Trainable params after freeze: 0  (must be 0)


In [50]:
# Print the full learned addition table
print("Learned addition table (a + b) mod 10:")
print("    b= 0  1  2  3  4  5  6  7  8  9")
for a in range(10):
    row = []
    for b in range(10):
        x = encode_pair(a, b).unsqueeze(0)
        with torch.no_grad():
            pred = model_a(x).argmax(1).item()
        row.append(str(pred))
    print(f"a={a}:  {' '.join(f'{v:>2}' for v in row)}")

Learned addition table (a + b) mod 10:
    b= 0  1  2  3  4  5  6  7  8  9
a=0:   0  1  2  3  4  5  6  7  8  9
a=1:   1  2  3  4  5  6  7  8  9  0
a=2:   2  3  4  5  6  7  8  9  0  1
a=3:   3  4  5  6  7  8  9  0  1  2
a=4:   4  5  6  7  8  9  0  1  2  3
a=5:   5  6  7  8  9  0  1  2  3  4
a=6:   6  7  8  9  0  1  2  3  4  5
a=7:   7  8  9  0  1  2  3  4  5  6
a=8:   8  9  0  1  2  3  4  5  6  7
a=9:   9  0  1  2  3  4  5  6  7  8


## Apply Frozen Model A to Adjacent Digit Pairs

apply the frozen addition unit to every adjacent pair for N digits.

input [3, 7, 5, 2] ----> [0, 2, 7]

In [51]:
def apply_addition_unit(digits: list, model: OnesDigitMLP) -> list:
    if len(digits) < 2:
        raise ValueError("Need at least 2 digits")

    outputs = []
    for i in range(len(digits) - 1):
        x = encode_pair(digits[i], digits[i + 1]).unsqueeze(0)
        with torch.no_grad():
            pred = model(x).argmax(1).item()
        outputs.append(pred)
    return outputs


def verify_m2(digits: list, model: OnesDigitMLP) -> None:
    #Pretty-print adjacent sums and confirm each matches ground truth.
    outputs = apply_addition_unit(digits, model)
    print(f"Input : {digits}")
    all_ok = True
    for i, (a, b, pred) in enumerate(zip(digits, digits[1:], outputs)):
        expected = (a + b) % 10
        ok = pred == expected
        status = "ok" if ok else f"X (expected {expected})"
        print(f"  [{i}]+[{i+1}]  {a}+{b} = {pred}  {status}")
        if not ok:
            all_ok = False
    print(f"Result: {'ALL CORRECT' if all_ok else 'ERRORS FOUND'}\n")

In [52]:
verify_m2([3, 7, 5, 2], model_a)   # spec example → [0, 2, 7]
verify_m2([0, 0, 0, 0], model_a)   # all zeros   → [0, 0, 0]
verify_m2([9, 9, 9, 9], model_a)   # wrap-around → [8, 8, 8]
verify_m2([1, 2, 3, 4, 5], model_a) # ascending  → [3, 5, 7, 9]
verify_m2([5, 5, 5], model_a)       # 5+5=0      → [0, 0]

Input : [3, 7, 5, 2]
  [0]+[1]  3+7 = 0  ok
  [1]+[2]  7+5 = 2  ok
  [2]+[3]  5+2 = 7  ok
Result: ALL CORRECT

Input : [0, 0, 0, 0]
  [0]+[1]  0+0 = 0  ok
  [1]+[2]  0+0 = 0  ok
  [2]+[3]  0+0 = 0  ok
Result: ALL CORRECT

Input : [9, 9, 9, 9]
  [0]+[1]  9+9 = 8  ok
  [1]+[2]  9+9 = 8  ok
  [2]+[3]  9+9 = 8  ok
Result: ALL CORRECT

Input : [1, 2, 3, 4, 5]
  [0]+[1]  1+2 = 3  ok
  [1]+[2]  2+3 = 5  ok
  [2]+[3]  3+4 = 7  ok
  [3]+[4]  4+5 = 9  ok
Result: ALL CORRECT

Input : [5, 5, 5]
  [0]+[1]  5+5 = 0  ok
  [1]+[2]  5+5 = 0  ok
Result: ALL CORRECT



## Trainable Attention + Frozen Model A

Replaces the manual adjacent pairing with a trainable attention mechanism.

In [53]:
class AdjacentAttentionAdder(nn.Module):
    def __init__(self, frozen_mlp: OnesDigitMLP, embed_dim: int = 32):
        super().__init__()
        self.frozen_mlp = frozen_mlp

        # Learned digit embedding
        self.embed = nn.Embedding(10, embed_dim)

        # Q and K projections for attention
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)

        # Relative position bias: one scalar per offset (-9 to +9 → indices 0..18)
        self.pos_bias = nn.Embedding(20, 1)

        # Small init so softmax doesn't saturate at start
        nn.init.normal_(self.W_q.weight, std=0.1)
        nn.init.normal_(self.W_k.weight, std=0.1)
        nn.init.zeros_(self.pos_bias.weight)

    def forward(self, digit_ids: torch.Tensor):
        B, N = digit_ids.shape
        x = self.embed(digit_ids)   # (B, N, E)

        Q = self.W_q(x)             # (B, N, E)
        K = self.W_k(x)             # (B, N, E)

        # Scaled dot-product scores
        scores = (Q @ K.transpose(-2, -1)) / (x.shape[-1] ** 0.5)   # (B, N, N)

        # Relative position bias
        pos = torch.arange(N, device=digit_ids.device)
        offsets = (pos.unsqueeze(0) - pos.unsqueeze(1)).clamp(-9, 9) + 9  # (N,N)
        scores = scores + self.pos_bias(offsets).squeeze(-1).unsqueeze(0)  # broadcast over B

        # Causal mask: position i can only attend to positions < i
        causal = torch.tril(torch.ones(N, N, dtype=torch.bool, device=digit_ids.device), diagonal=-1)
        causal[0, 0] = True   # pos 0 attends to itself to avoid nan in softmax
        scores = scores.masked_fill(~causal.unsqueeze(0), float('-inf'))

        attn = F.softmax(scores, dim=-1)   # (B, N, N)

        # Retrieve: weighted combination of digit one-hots
        digit_onehots = F.one_hot(digit_ids, num_classes=10).float()   # (B, N, 10)
        retrieved = attn @ digit_onehots    # (B, N, 10)  — soft digit representation

        # Build input for frozen MLP: [retrieved_prev | current]
        retrieved_prev = retrieved[:, 1:, :]        # (B, N-1, 10)
        current        = digit_onehots[:, 1:, :]    # (B, N-1, 10)
        pair_input = torch.cat([retrieved_prev, current], dim=-1)   # (B, N-1, 20)

        # Run through frozen Model A
        logits = self.frozen_mlp(
            pair_input.reshape(B * (N - 1), 20)
        ).reshape(B, N - 1, 10)

        return logits, attn

In [54]:
def train_m3(model_a: OnesDigitMLP, max_epochs: int = 5_000, lr: float = 1e-2) -> AdjacentAttentionAdder:
    #Train the attention block. Model A is never updated.
    torch.manual_seed(42)
    block = AdjacentAttentionAdder(frozen_mlp=model_a)
    optimizer = optim.Adam(block.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    # Confirm Model A is frozen
    assert sum(p.numel() for p in model_a.parameters() if p.requires_grad) == 0

    print("Training attention block (Model A stays frozen)...")
    for epoch in range(1, max_epochs + 1):
        block.train()
        # Random sequences of length 8, batch size 512
        seqs = torch.randint(0, 10, (512, 8))
        targets = torch.stack(
            [(seqs[:, i] + seqs[:, i + 1]) % 10 for i in range(7)], dim=1
        )
        logits, _ = block(seqs)
        loss = F.cross_entropy(logits.reshape(-1, 10), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if epoch % 500 == 0:
            with torch.no_grad():
                acc = (logits.argmax(-1) == targets).float().mean().item()
            print(f"  epoch {epoch:5d}  loss={loss.item():.4f}  acc={acc:.2%}")
            if acc == 1.0:
                print(f"  100% accuracy reached at epoch {epoch}!")
                break

    block.eval()
    return block

attention_block = train_m3(model_a)

Training attention block (Model A stays frozen)...
  epoch   500  loss=0.5587  acc=99.61%
  epoch  1000  loss=0.5492  acc=99.97%
  epoch  1500  loss=0.5474  acc=100.00%
  100% accuracy reached at epoch 1500!


In [55]:
def verify_m3(block: AdjacentAttentionAdder, digits: list) -> torch.Tensor:
    t = torch.tensor([digits])
    with torch.no_grad():
        logits, attn = block(t)
    preds    = logits.argmax(-1).squeeze(0).tolist()
    expected = [(digits[i] + digits[i+1]) % 10 for i in range(len(digits)-1)]
    ok = "ok" if preds == expected else "X"
    print(f"  {digits} → {preds}  expected={expected}  {ok}")
    return attn.squeeze(0)

print("Verification:")
attn = verify_m3(attention_block, [3, 7, 5, 2])
verify_m3(attention_block, [9, 9, 9, 9])
verify_m3(attention_block, [1, 2, 3, 4, 5])
verify_m3(attention_block, [5, 5, 5])
verify_m3(attention_block, [1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 9, 9])


print("\nAttention weights for [3, 7, 5, 2]:")
print("(Row i shows what position i attends to. Expect strong weight at column i-1.)")
print(f"{'':8s}" + "  ".join(f"pos{j}" for j in range(4)))
for i, row in enumerate(attn):
    vals = "  ".join(f"{w:.2f}" for w in row.tolist())
    print(f"  pos {i}:  [{vals}]")

Verification:
  [3, 7, 5, 2] → [0, 2, 7]  expected=[0, 2, 7]  ok
  [9, 9, 9, 9] → [8, 8, 8]  expected=[8, 8, 8]  ok
  [1, 2, 3, 4, 5] → [3, 5, 7, 9]  expected=[3, 5, 7, 9]  ok
  [5, 5, 5] → [0, 0]  expected=[0, 0]  ok
  [1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 9, 9] → [3, 5, 7, 9, 1, 3, 5, 7, 9, 9, 8]  expected=[3, 5, 7, 9, 1, 3, 5, 7, 9, 9, 8]  ok

Attention weights for [3, 7, 5, 2]:
(Row i shows what position i attends to. Expect strong weight at column i-1.)
        pos0  pos1  pos2  pos3
  pos 0:  [1.00  0.00  0.00  0.00]
  pos 1:  [1.00  0.00  0.00  0.00]
  pos 2:  [0.00  1.00  0.00  0.00]
  pos 3:  [0.00  0.00  0.99  0.00]


## Train & Freeze Model B (Full Adder with Carry)

Given a, b plus a carry-in bit predict the ones and carry digit


In [56]:
class FullAdderMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.h1         = nn.Linear(22, 64)
        self.h2         = nn.Linear(64, 64)
        self.ones_head  = nn.Linear(64, 10)
        self.carry_head = nn.Linear(64, 2)

    def forward(self, x: torch.Tensor):
        h = F.relu(self.h2(F.relu(self.h1(x))))
        return self.ones_head(h), self.carry_head(h)


def train_model_b(seed: int = 0, max_epochs: int = 20_000, lr: float = 1e-2, verbose: bool = True):
    #Train Model B to 100% accuracy on all 200 combinations.
    torch.manual_seed(seed)
    model = FullAdderMLP()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    t0 = time.time()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        lo, lc = model(X200)
        loss = F.cross_entropy(lo, Y_ones200) + F.cross_entropy(lc, Y_carry200)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            ones_ok  = model(X200)[0].argmax(1).eq(Y_ones200).all()
            carry_ok = model(X200)[1].argmax(1).eq(Y_carry200).all()
            if ones_ok and carry_ok:
                if verbose:
                    print(f"  Model B: 100% at epoch {epoch}")
                break

    elapsed = time.time() - t0
    for p in model.parameters():
        p.requires_grad_(False)
    return model.eval(), epoch, elapsed


model_b, _, _ = train_model_b()

# Verify
with torch.no_grad():
    lo, lc = model_b(X200)
print(f"Model B ones accuracy : {lo.argmax(1).eq(Y_ones200).sum().item()}/200")
print(f"Model B carry accuracy: {lc.argmax(1).eq(Y_carry200).sum().item()}/200")

  Model B: 100% at epoch 50
Model B ones accuracy : 200/200
Model B carry accuracy: 200/200


In [57]:
# Print full learned truth table (first 20 rows as sample)
print("Full adder truth table (a + b + cin):")
print(f"{'a':>2} {'b':>2} {'cin':>4}  →  {'ones':>4} {'carry':>5}  {'correct?':>8}")
print("-" * 40)
with torch.no_grad():
    lo, lc = model_b(X200)
pred_ones  = lo.argmax(1)
pred_carry = lc.argmax(1)
errors = 0
for idx, (a, b, cin) in enumerate([(a,b,c) for a in range(10) for b in range(10) for c in range(2)]):
    po = pred_ones[idx].item(); pc = pred_carry[idx].item()
    eo = Y_ones200[idx].item(); ec = Y_carry200[idx].item()
    ok = (po == eo) and (pc == ec)
    if not ok: errors += 1
    if idx < 20 or not ok:
        print(f"{a:>2} {b:>2} {cin:>4}  →  {po:>4} {pc:>5}  {'ok' if ok else 'X'}")
print(f"\nTotal errors: {errors}/200")

Full adder truth table (a + b + cin):
 a  b  cin  →  ones carry  correct?
----------------------------------------
 0  0    0  →     0     0  ok
 0  0    1  →     1     0  ok
 0  1    0  →     1     0  ok
 0  1    1  →     2     0  ok
 0  2    0  →     2     0  ok
 0  2    1  →     3     0  ok
 0  3    0  →     3     0  ok
 0  3    1  →     4     0  ok
 0  4    0  →     4     0  ok
 0  4    1  →     5     0  ok
 0  5    0  →     5     0  ok
 0  5    1  →     6     0  ok
 0  6    0  →     6     0  ok
 0  6    1  →     7     0  ok
 0  7    0  →     7     0  ok
 0  7    1  →     8     0  ok
 0  8    0  →     8     0  ok
 0  8    1  →     9     0  ok
 0  9    0  →     9     0  ok
 0  9    1  →     0     1  ok

Total errors: 0/200


In [58]:
class ModelC(nn.Module):
    def __init__(self, pretrained_a: OnesDigitMLP):
        super().__init__()
        self.h1         = nn.Linear(20, 64)   # same as Model A's h1 (will be frozen)
        self.h2         = nn.Linear(64, 64)
        self.ones_head  = nn.Linear(64, 10)
        self.carry_head = nn.Linear(64, 2)

        # Copy Model A's first layer weights
        with torch.no_grad():
            self.h1.weight.copy_(pretrained_a.h1.weight)
            self.h1.bias.copy_(pretrained_a.h1.bias)

        # Freeze the first hidden layer
        for p in self.h1.parameters():
            p.requires_grad_(False)

    def forward(self, x: torch.Tensor):
        # Model C only sees 20-d input (no carry_in — tests transfer on same task)
        h = F.relu(self.h2(F.relu(self.h1(x))))
        return self.ones_head(h), self.carry_head(h)


def train_model_c(pretrained_a: OnesDigitMLP, seed: int = 0,
                  max_epochs: int = 20_000, lr: float = 1e-2):
    #Train Model C (frozen h1 from A) on ones + carry for 100 pairs.
    torch.manual_seed(seed)
    Y_carry_100 = torch.tensor(
        [(a + b) // 10 for a in range(10) for b in range(10)], dtype=torch.long
    )
    model = ModelC(pretrained_a)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr)
    t0 = time.time()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        lo, lc = model(X100)
        loss = F.cross_entropy(lo, Y_ones) + F.cross_entropy(lc, Y_carry_100)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            ones_ok  = model(X100)[0].argmax(1).eq(Y_ones).all()
            carry_ok = model(X100)[1].argmax(1).eq(Y_carry_100).all()
            if ones_ok and carry_ok:
                break

    return model.eval(), epoch, time.time() - t0


# 10-trial comparison 
TRIALS = 10
b_epochs, b_times = [], []
c_epochs, c_times = [], []

print(f"Running {TRIALS}-trial comparison: Model B (scratch) vs Model C (frozen h1 from A)\n")
for trial in range(TRIALS):
    _, ep, t = train_model_b(seed=trial, verbose=False)
    b_epochs.append(ep); b_times.append(t)
    _, ep, t = train_model_c(model_a, seed=trial)
    c_epochs.append(ep); c_times.append(t)

print("=" * 42)
print(f"Model B  mean epochs: {sum(b_epochs)/TRIALS:7.1f}  mean time: {sum(b_times)/TRIALS:.3f}s")
print(f"Model C  mean epochs: {sum(c_epochs)/TRIALS:7.1f}  mean time: {sum(c_times)/TRIALS:.3f}s")
print("=" * 42)
winner = "C (frozen transfer)" if sum(c_times) < sum(b_times) else "B (scratch)"
speedup = abs(sum(b_times) - sum(c_times)) / sum(b_times) * 100
print(f"Winner: {winner}  (faster by {speedup:.1f}%)")
print()
print("Interpretation: Does freezing Model A's first layer speed up learning?")
print(f"  {'YES — the frozen representation gives a useful head start.' if sum(c_times) < sum(b_times) else 'NO — Model B from scratch is faster in this configuration.'}")

Running 10-trial comparison: Model B (scratch) vs Model C (frozen h1 from A)

Model B  mean epochs:    56.4  mean time: 0.102s
Model C  mean epochs:    27.0  mean time: 0.040s
Winner: C (frozen transfer)  (faster by 60.2%)

Interpretation: Does freezing Model A's first layer speed up learning?
  YES — the frozen representation gives a useful head start.


## Stack Frozen Units into a Two-Digit Adder

Ties frozen Model A and Model B together to perform full two-digit addition.

The carry output from the ones column is routed as carry input to the tens column.
No gradient flows through the frozen modules.

In [59]:
class TwoDigitAdder(nn.Module):
    def __init__(self, model_a: OnesDigitMLP, model_b: FullAdderMLP):
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b
        # Ensure both are frozen
        for p in self.model_a.parameters(): p.requires_grad_(False)
        for p in self.model_b.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def forward(self, a: int, b: int) -> dict:
        a_ones = a % 10;       b_ones = b % 10
        a_tens = (a // 10) % 10; b_tens = (b // 10) % 10

        # Ones column: Model A 
        x_ones = torch.zeros(1, 22)
        x_ones[0, a_ones]    = 1.0
        x_ones[0, 10+b_ones] = 1.0
        x_ones[0, 20+0]      = 1.0    # no carry into ones column
        _, carry_logits = self.model_b(x_ones)
        carry_out_ones = carry_logits.argmax(1).item()

        x_a = torch.zeros(1, 20)
        x_a[0, a_ones]    = 1.0
        x_a[0, 10+b_ones] = 1.0
        ones_digit = self.model_a(x_a).argmax(1).item()

        # Tens column: Model B (with carry from ones) 
        x_tens = torch.zeros(1, 22)
        x_tens[0, a_tens]           = 1.0
        x_tens[0, 10+b_tens]        = 1.0
        x_tens[0, 20+carry_out_ones] = 1.0   # carry from ones column
        tens_logits, carry_logits_tens = self.model_b(x_tens)
        tens_digit    = tens_logits.argmax(1).item()
        carry_out_final = carry_logits_tens.argmax(1).item()

        result = tens_digit * 10 + ones_digit
        return {
            "ones_digit":   ones_digit,
            "tens_digit":   tens_digit,
            "carry_out":    carry_out_final,
            "result":       result
        }


adder = TwoDigitAdder(model_a, model_b)

In [60]:
# Spot checks
print("Spot checks:")
tests = [(27,58,85), (99,99,98), (50,50,0), (13,29,42), (0,0,0), (9,1,10%100)]
for a, b, expected in tests:
    r = adder(a, b)
    ok = "ok" if r["result"] == expected else "X"
    print(f"  {a:02d} + {b:02d} = {r['result']:02d}  (expected {expected:02d})  {ok}")

# Full sweep: all 10,000 combinations
print("\nFull sweep (all 10,000 combinations)...")
correct = sum(adder(a, b)["result"] == (a+b) % 100
              for a in range(100) for b in range(100))
print(f"Accuracy: {correct}/10000 ")

Spot checks:
  27 + 58 = 85  (expected 85)  ok
  99 + 99 = 98  (expected 98)  ok
  50 + 50 = 00  (expected 00)  ok
  13 + 29 = 42  (expected 42)  ok
  00 + 00 = 00  (expected 00)  ok
  09 + 01 = 10  (expected 10)  ok

Full sweep (all 10,000 combinations)...
Accuracy: 10000/10000 


## To Do: Build a transformer that takes two multi-digit numbers as a token sequence and produces their digit-by-digit sum, where all arithmetic is handled by the frozen subunits. 